In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("Libraries imported successfully.")

In [ ]:
# Advanced Data Profiler: Patient Health Records
**Author:** [Your Name]
**Objective:** Perform advanced preprocessing, feature engineering, and exploratory data analysis on real patient health records to predict hospital readmission.

## Part A: Fundamentals
**1. What is Data Analysis?**
Data Analysis is the systematic process of inspecting, cleansing, transforming, and modeling data to uncover patterns, identify anomalies, and support clinical decision-making.

**2. Planning a Data Science Project:**
1. Problem Definition & Clinical Understanding
2. Data Acquisition (CSV, SQL, API, JSON)
3. Data Cleaning & Preprocessing (Imputation, Outlier Handling)
4. Exploratory Data Analysis (Univariate, Bivariate, Multivariate)
5. Feature Engineering (Encoding, Scaling, Creating new features)
6. Model Building & Evaluation
7. Deployment & Clinical Monitoring

**3. ML Problem Statement:**
*Predict whether a patient will require follow-up/readmission based on their health records.*
- **Input:** Clinical data (Age, BP, BMI, Glucose, Diagnosis, etc.)
- **Output:** Multi-class Classification (Discharged, Follow-up Required, Transferred).
- **Type:** Supervised Learning (Classification).

**4. Tensors:**
Tensors are multi-dimensional arrays. 
- **0D Tensor:** Scalar (e.g., 5)
- **1D Tensor:** Vector (e.g., [120, 80, 90])
- **2D Tensor:** Matrix (e.g., [[120, 80], [130, 85]])
- **3D Tensor:** Higher dimensional arrays.

In [ ]:
print("--- PART B: ADVANCED DATA ACQUISITION ---")

# 1. Load the main CSV file
df_csv = pd.read_csv('patient_health_records.csv')
print(f"1. Main CSV loaded successfully. Shape: {df_csv.shape}")

# 2. Parse JSON (Simulating a JSON payload from a wearable device)
json_data = json.dumps({
    "Patient_ID": ["PAT-9999"], "Age": [45], "Gender": ["Male"], 
    "Blood_Type": ["O+"], "Primary_Diagnosis": ["Hypertension"], 
    "Systolic_BP_mmHg": [135], "Diastolic_BP_mmHg": [88], 
    "Heart_Rate_bpm": [78], "Fasting_Glucose_mgdL": [110.0], 
    "BMI": [28.5], "SpO2_Percent": [98], 
    "Prescribed_Medication": ["Lisinopril"], 
    "Admission_Type": ["Routine Checkup"], 
    "Visit_Date": ["2026-09-15"], "Discharge_Status": ["Discharged"]
})
df_json = pd.read_json(json_data)
print("2. JSON parsed successfully.")

# 3. Connect to SQL (SQLite in-memory database)
conn = sqlite3.connect(':memory:')
df_csv.to_sql('patients', conn, index=False, if_exists='replace')
df_sql = pd.read_sql_query("SELECT * FROM patients LIMIT 5", conn)
print("3. SQL connection and fetch successful.")

# 4. Fetch from API (Simulating an external lab API)
api_response = [{
    "Patient_ID": "PAT-8888", "Age": 60, "Gender": "Female", 
    "Blood_Type": "A-", "Primary_Diagnosis": "Type 2 Diabetes", 
    "Systolic_BP_mmHg": 145, "Diastolic_BP_mmHg": 92, 
    "Heart_Rate_bpm": 88, "Fasting_Glucose_mgdL": 180.0, 
    "BMI": 31.0, "SpO2_Percent": 95, 
    "Prescribed_Medication": "Metformin", 
    "Admission_Type": "Urgent", "Visit_Date": "2026-09-14", 
    "Discharge_Status": "Follow-up Required"
}]
df_api = pd.DataFrame(api_response)
print("4. API fetch successful.")

# Combine all sources into a master DataFrame
master_df = pd.concat([df_csv, df_json, df_api], ignore_index=True)
print(f"\n Master Dataset Combined. Shape: {master_df.shape}")

In [ ]:
print("--- PART C: DATA CLEANING & FEATURE ENGINEERING ---")

# 5. Initial Exploration
print("\n--- Dataset Info ---")
print(master_df.info())
print("\n--- Dataset Describe ---")
print(master_df.describe(include='all'))

# 6. Handle Missing Values (Check first)
print(f"\nMissing values before cleaning:\n{master_df.isnull().sum()}")

# 7. Data Cleaning
# a. Remove duplicates
master_df.drop_duplicates(subset=['Patient_ID'], keep='first', inplace=True)

# b. Convert Visit_Date to datetime
master_df['Visit_Date'] = pd.to_datetime(master_df['Visit_Date'])

# c. Handle missing values (if any existed in your CSV, we fill them)
# For numeric columns: fill with median
numeric_cols = ['Age', 'Systolic_BP_mmHg', 'Diastolic_BP_mmHg', 'Heart_Rate_bpm', 'Fasting_Glucose_mgdL', 'BMI', 'SpO2_Percent']
for col in numeric_cols:
    master_df[col] = master_df[col].fillna(master_df[col].median())

# For categorical columns: fill with mode
cat_cols = ['Gender', 'Blood_Type', 'Primary_Diagnosis', 'Prescribed_Medication', 'Admission_Type', 'Discharge_Status']
for col in cat_cols:
    master_df[col] = master_df[col].fillna(master_df[col].mode()[0])

print(f"\nMissing values after cleaning:\n{master_df.isnull().sum()}")

# 8. Advanced Feature Engineering
# a. Calculate Mean Arterial Pressure (MAP)
master_df['MAP'] = (master_df['Systolic_BP_mmHg'] + 2 * master_df['Diastolic_BP_mmHg']) / 3

# b. Create Hypertension Risk Category based on BP
def bp_category(row):
    if row['Systolic_BP_mmHg'] >= 140 or row['Diastolic_BP_mmHg'] >= 90:
        return 'Hypertensive'
    elif row['Systolic_BP_mmHg'] >= 120:
        return 'Elevated'
    else:
        return 'Normal'
master_df['BP_Category'] = master_df.apply(bp_category, axis=1)

# c. Create Age Groups
bins = [0, 30, 50, 70, 100]
labels = ['Young', 'Middle-aged', 'Senior', 'Elderly']
master_df['Age_Group'] = pd.cut(master_df['Age'], bins=bins, labels=labels)

# d. Encode Categorical Variables for ML
le = LabelEncoder()
encode_cols = ['Gender', 'Blood_Type', 'Primary_Diagnosis', 'Prescribed_Medication', 
               'Admission_Type', 'Discharge_Status', 'BP_Category', 'Age_Group']
for col in encode_cols:
    master_df[col + '_Encoded'] = le.fit_transform(master_df[col].astype(str))

print("\n Data Cleaning and Feature Engineering Complete!")
print(f"New features added: MAP, BP_Category, Age_Group, and Encoded columns.")

In [ ]:
print("--- PART D: EXPLORATORY DATA ANALYSIS (EDA) ---")

# 9. Univariate Analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
sns.histplot(master_df['Age'], kde=True, ax=axes[0,0], color='blue').set_title('Age Distribution')
sns.histplot(master_df['BMI'], kde=True, ax=axes[0,1], color='green').set_title('BMI Distribution')
sns.histplot(master_df['Fasting_Glucose_mgdL'], kde=True, ax=axes[0,2], color='orange').set_title('Glucose Distribution')
sns.histplot(master_df['Systolic_BP_mmHg'], kde=True, ax=axes[1,0], color='red').set_title('Systolic BP')
sns.histplot(master_df['Heart_Rate_bpm'], kde=True, ax=axes[1,1], color='purple').set_title('Heart Rate')
sns.histplot(master_df['SpO2_Percent'], kde=True, ax=axes[1,2], color='brown').set_title('SpO2 Levels')
plt.tight_layout()
plt.show()

# 10. Bivariate Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(x='Discharge_Status', y='Age', data=master_df, ax=axes[0]).set_title('Age vs Discharge Status')
sns.boxplot(x='Primary_Diagnosis', y='Fasting_Glucose_mgdL', data=master_df, ax=axes[1]).set_title('Glucose by Diagnosis')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')
sns.boxplot(x='Admission_Type', y='BMI', data=master_df, ax=axes[2]).set_title('BMI by Admission Type')
plt.tight_layout()
plt.show()

# 11. Multivariate Analysis
plt.figure(figsize=(12, 8))
numeric_df = master_df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap of Medical Variables')
plt.show()

# Pairplot for key features
sns.pairplot(master_df[['Age', 'BMI', 'Fasting_Glucose_mgdL', 'Systolic_BP_mmHg', 'Discharge_Status']], 
             hue='Discharge_Status', diag_kind='kde', plot_kws={'alpha': 0.6})
plt.show()